# 🏠 Housing Price Prediction — Colab Playground

נוטבוק עצמאי להרצת המודל ב-Colab.

**זרימה**:
1. Install dependencies
2. העלה את `temporal_features.xlsx` + `features_osm.csv`
3. Load & explore data
4. Train LightGBM
5. Evaluate
6. Feature importance + predictions
7. Play cells (חיזוי בודד, segmented metrics, וכו')

## 1. Install dependencies

In [ ]:
!pip install -q lightgbm openpyxl joblib

## 2. Upload data files

הרץ את התא וגרור/בחר את שני הקבצים:
- `temporal_features.xlsx`
- `features_osm.csv`

אם אתה רץ לוקאלי ולא ב-Colab — דלג על התא הזה והשים את הקבצים באותה תיקייה.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))
except ImportError:
    print('Not in Colab — place files manually in working dir.')

### אופציה חלופית: Google Drive
אם הקבצים אצלך ב-Drive, הרץ את זה במקום ההעלאה הידנית:

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/housing_data'  # ← שנה לנתיב שלך
# import shutil
# shutil.copy(f'{DATA_DIR}/temporal_features.xlsx', '.')
# shutil.copy(f'{DATA_DIR}/features_osm.csv', '.')

## 3. Data loading layers (inline — אותה לוגיקה כמו `common/data.py`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TARGET = 'real_price'

LEAKAGE_COLS = {
    'price', 'price_per_sqm', 'real_price', 'real_price_per_sqm',
    'real_price_factor', 'log_price', 'log_price_per_sqm', 'log_real_price',
}
ID_COLS = {'_id', 'source_name', 'city', 'neighborhood', 'street',
           'project_name', 'transaction_date'}
CATEGORICAL_COLS = ['deal_nature']

TRAIN_END = pd.Timestamp('2024-01-01')
VAL_END = pd.Timestamp('2025-01-01')
MIN_DATE = pd.Timestamp('2015-01-01')
MIN_AREA, MAX_AREA = 15, 500
WINSOR_LOW, WINSOR_HIGH = 0.01, 0.99


def load_temporal(path='temporal_features.xlsx'):
    df = pd.read_excel(path, sheet_name='transactions')
    df['_id'] = df['_id'].astype(str)
    df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
    return df

def load_osm(path='features_osm.csv'):
    df = pd.read_csv(path)
    df['_id'] = df['_id'].astype(str)
    df = df.drop(columns=[c for c in ('feature_source', 'feature_version') if c in df.columns])
    return df

def join_layers(temporal, osm):
    return temporal.merge(osm, on='_id', how='left', suffixes=('', '_osm'))

def clean(df):
    # LightGBM מטפל ב-NaN — מפילים רשומה רק אם חסר target או date.
    df = df.copy()
    df = df[df[TARGET].notna()]
    df = df[df['transaction_date'].notna()]
    df = df[df['transaction_date'] >= MIN_DATE]
    area_ok = df['area_sqm'].isna() | df['area_sqm'].between(MIN_AREA, MAX_AREA)
    df = df[area_ok]
    lo, hi = df[TARGET].quantile([WINSOR_LOW, WINSOR_HIGH])
    df = df[df[TARGET].between(lo, hi)]
    return df.reset_index(drop=True)

def select_features(df):
    drop = LEAKAGE_COLS | ID_COLS | {TARGET}
    X = df.drop(columns=[c for c in drop if c in df.columns])
    for c in CATEGORICAL_COLS:
        if c in X.columns:
            X[c] = X[c].astype('category')
    return X

def split(df):
    train_mask = df['transaction_date'] < TRAIN_END
    val_mask = (df['transaction_date'] >= TRAIN_END) & (df['transaction_date'] < VAL_END)
    test_mask = df['transaction_date'] >= VAL_END
    def xy(mask):
        sub = df[mask]
        return select_features(sub), sub[TARGET].values.astype(float)
    return (*xy(train_mask), *xy(val_mask), *xy(test_mask))

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    err = y_pred - y_true
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    mask = y_true > 0
    mape = float(np.mean(np.abs(err[mask]/y_true[mask]))) if mask.any() else float('nan')
    ss_res = float(np.sum(err**2))
    ss_tot = float(np.sum((y_true - y_true.mean())**2))
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else float('nan')
    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'n': int(len(y_true))}

print('✅ loaders defined')


## 4. Explore — שכבה אחרי שכבה

In [ ]:
temporal = load_temporal()
print('temporal:', temporal.shape)
temporal.head()

In [ ]:
osm = load_osm()
print('osm:', osm.shape)
osm.head()

In [ ]:
merged = join_layers(temporal, osm)
coverage = merged[osm.columns.drop('_id')[0]].notna().mean()
print(f'merged: {len(merged):,}   OSM coverage: {coverage:.1%}')

In [ ]:
cleaned = clean(merged)
print(f'after clean: {len(cleaned):,}')
print(f'date range:  {cleaned["transaction_date"].min()} → {cleaned["transaction_date"].max()}')

In [ ]:
# התפלגות target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cleaned['real_price'].hist(bins=100, ax=axes[0])
axes[0].set_title('real_price')
np.log1p(cleaned['real_price']).hist(bins=100, ax=axes[1])
axes[1].set_title('log(1 + real_price)')
plt.tight_layout(); plt.show()

In [ ]:
# התפלגות מחירים לפי שנה
cleaned.groupby(cleaned['transaction_date'].dt.year)['real_price'].median().plot(kind='bar', figsize=(10, 4))
plt.title('median real_price by year'); plt.tight_layout(); plt.show()

## 5. Split

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = split(cleaned)
print(f'train: {len(X_train):>7,}')
print(f'val:   {len(X_val):>7,}')
print(f'test:  {len(X_test):>7,}')
print(f'features: {X_train.shape[1]}')
list(X_train.columns)

## 6. Train LightGBM

In [ ]:
import lightgbm as lgb

PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model = lgb.LGBMRegressor(**PARAMS)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='mape',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

## 7. Evaluate

In [ ]:
y_pred = model.predict(X_test)
metrics = compute_metrics(y_test, y_pred)
for k, v in metrics.items():
    print(f'{k:>6}: {v:,.4f}' if isinstance(v, float) else f'{k:>6}: {v:,}')

In [ ]:
# Predicted vs Actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.1, s=5)
lim = [0, max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, 'r--', lw=1)
plt.xlabel('actual'); plt.ylabel('predicted'); plt.title('Predicted vs Actual (test)')
plt.tight_layout(); plt.show()

In [ ]:
# Residuals
residuals = y_pred - y_test
plt.figure(figsize=(10, 4))
plt.hist(residuals, bins=100)
plt.axvline(0, color='r', lw=1)
plt.title(f'Residuals (mean={residuals.mean():,.0f}, std={residuals.std():,.0f})')
plt.tight_layout(); plt.show()

## 8. Feature importance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
lgb.plot_importance(model, max_num_features=30, importance_type='gain', ax=ax)
plt.tight_layout(); plt.show()

## 9. Segmented metrics (לפי עיר)

In [ ]:
test_df = cleaned[cleaned['transaction_date'] >= VAL_END].copy().reset_index(drop=True)
test_df['y_pred'] = y_pred

city_stats = []
for city, g in test_df.groupby('city'):
    if len(g) < 50: continue
    m = compute_metrics(g['real_price'], g['y_pred'])
    city_stats.append({'city': city, 'n': m['n'], 'mape': m['mape'], 'mae': m['mae']})

pd.DataFrame(city_stats).sort_values('mape').head(20)

## 10. Single prediction (playground)

In [ ]:
# קח שורה אחת מה-test ובחן:
i = 0
row = X_test.iloc[[i]]
actual = y_test[i]
predicted = model.predict(row)[0]
print(f'actual:    {actual:>12,.0f} ₪')
print(f'predicted: {predicted:>12,.0f} ₪')
print(f'error:     {predicted - actual:>12,.0f} ₪  ({(predicted-actual)/actual:.1%})')
row.T

## 11. Save model

In [ ]:
import joblib
joblib.dump(model, 'lightgbm_v1.joblib')

# ב-Colab — להוריד לוקאלית:
try:
    from google.colab import files
    files.download('lightgbm_v1.joblib')
except ImportError:
    print('saved to lightgbm_v1.joblib')

## 🎮 Play zone — נסיונות חופשיים

תאים ריקים לשחק. דוגמאות:
- שנה hyperparameter ואמן מחדש
- הוסף feature חדש ל-`cleaned` ורץ מחדש
- בדוק ביצועים לפי טווחי מחיר
- נסה מודל אחר (XGBoost, Linear...)